# Determinants of Firm Growth: A Heckman (Tobit II) Approach

Companion notebook to *Informative Missingness, Firm Risk, and Zombie Firms* -- same underlying data, a different research question. Instead of predicting **whether** a firm fails, this notebook asks: **among the determinants of firm growth, which ones matter, once we correct for the fact that firms that exit never get to show a growth rate at all?**

This is the textbook selection-bias problem in the firm-growth literature (Evans, 1987; Hall, 1987): if you regress growth on size, age, productivity, etc. using only surviving firms, you silently condition on survival. Firms whose (unobserved) prospects were poor enough to make them exit are missing from the growth regression entirely -- and if the same unobserved factors that drive exit also drive growth conditional on survival, ordinary least squares on survivors is biased. The **Heckman two-step estimator** (a special case of the **Tobit II** model) corrects for this with a two-equation system:

1. **Selection equation** (Probit): does the firm survive to have a growth rate observed at all?
2. **Outcome equation** (OLS): among survivors, what predicts growth -- augmented with a correction term (the *inverse Mills ratio*) built from the selection equation, so that the two equations' error terms are allowed to be correlated.

We start with Italy and measure growth as **log employment growth** from year `t` to `t+1`, built from the same wide panel (`data_wide_IT.csv`) used in the companion notebook.


## How this notebook is organized

1. **Setup** (secs 1-2): packages, data, country choice.
2. **Building the growth panel** (sec 3): stack year-pairs `(t, t+1)` into one long panel, define the growth measure and the selection indicator.
3. **What "selection" really means here** (sec 4): a first look at attrition, and a direct link back to the companion notebook's missingness findings -- this matters a lot for how to read everything that follows.
4. **Selection equation** (sec 5): a Probit model for "does this firm survive to grow?", including a peer-group exclusion restriction.
5. **Naive vs. Heckman-corrected growth regression** (sec 6): the biased survivors-only OLS, then the two-step correction, side by side.
6. **Standard errors** (sec 7): why the naive two-step standard errors are not quite right, and what to do about it.
7. **Limitations and extensions** (sec 8): what this analysis still can't tell you, and how you'd take it further (other countries, sales growth, a fully missing-aware selection stage).


## 1. Colab environment


In [1]:
%pip install -q statsmodels xgboost linearmodels


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.3 MB/s eta 0:00:00


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy.stats import norm
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from xgboost import XGBClassifier
from linearmodels.iv import IV2SLS

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Configuration

Same `Dati_CSV` folder as the companion notebook. This version works from `data_wide_{COUNTRY}.csv`, which has one row per firm with year-by-year financials -- exactly what's needed to build `(t, t+1)` growth pairs. Starting with `COUNTRY = "IT"` (the curated sample); the panel-building code below is written so that switching country later is mostly a matter of re-running it, but FR/PT/ES come from the rawer multi-country panel (see the companion notebook's section 2), so expect smaller samples and check completeness before trusting the results.


In [3]:
USE_GOOGLE_DRIVE = True
DATA_DIR = Path("/content/drive/MyDrive/Zombie/Dati_CSV")

COUNTRY = "IT"
WIDE_FILE = DATA_DIR / f"data_wide_{COUNTRY}.csv"

YEARS = list(range(2008, 2017))
GROWTH_YEARS = range(2008, 2016)  # t -> t+1, so the last t is 2015 (employees_2017 doesn't exist)

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if not WIDE_FILE.exists():
    raise FileNotFoundError(f"File not found: {WIDE_FILE}")


Mounted at /content/drive


## 3. Building the growth panel

`data_wide_{COUNTRY}.csv` has one row per firm with separate columns per year (e.g. `employees_2008`, ..., `employees_2016`). To study growth we need to turn this into a **stacked panel**: one row per *(firm, year pair)*, where `t` is the starting year and `t+1` the year growth is measured to. We reuse the same `year_col()` naming-convention helper and covariate list as the companion notebook, restricted to variables that plausibly matter for growth (financial constraints, productivity, distress signals, innovation) rather than the full predictor set used for failure prediction.


In [4]:
def year_col(base, year):
    if base in {"depr", "revenue"}:
        return f"{base}{year}"
    if base == "misallocated_fixed":
        return f"misallocated_{year}_fixed"
    return f"{base}_{year}"


# Growth determinants from the literature: financial constraints (liquidity_ratio,
# solvency_ratio, real_SA, fin_cons100), productivity (tfp_acf), and distress signals
# (ICR_failure, NEG_VA, profitability) that a Tobit II model can test as predictors of
# BOTH survival and, conditional on survival, growth.
COVARIATE_BASES = ["tfp_acf", "liquidity_ratio", "solvency_ratio", "real_SA",
                    "ICR_failure", "NEG_VA", "profitability", "fin_cons100"]

wide = pd.read_csv(WIDE_FILE, low_memory=False)

# year_of_incorporation is not in older versions of data_wide_IT.csv: merge it in memory from the
# patch file in the same Drive folder (the data file on Drive is left untouched).
PATCH_FILE = DATA_DIR / "year_of_incorporation_patch.csv"
if "year_of_incorporation" not in wide.columns:
    if not PATCH_FILE.exists():
        raise FileNotFoundError(f"'year_of_incorporation' missing from {WIDE_FILE.name} and patch not found: {PATCH_FILE}")
    wide = wide.merge(pd.read_csv(PATCH_FILE, usecols=["id", "year_of_incorporation"]), on="id", how="left")
    print(f"Merged year_of_incorporation from {PATCH_FILE.name}: "
          f"{wide['year_of_incorporation'].notna().mean():.1%} of firms matched")

panels = []
for t in GROWTH_YEARS:
    emp_t, emp_t1 = year_col("employees", t), year_col("employees", t + 1)
    fail_t1 = f"failure_{t + 1}"
    cols = [emp_t, emp_t1, fail_t1] + [year_col(b, t) for b in COVARIATE_BASES]
    if any(c not in wide.columns for c in cols):
        continue  # this year pair isn't fully available in this country's file
    sub = wide[["id", "nace_2", "region", "dummy_patents", "dummy_trademark",
                "year_of_incorporation"] + cols].copy()
    sub = sub.rename(columns={emp_t: "employees_t", emp_t1: "employees_t1", fail_t1: "failure_t1"})
    sub = sub.rename(columns={year_col(b, t): b for b in COVARIATE_BASES})
    sub["year"] = t
    panels.append(sub)

panel = pd.concat(panels, ignore_index=True)
panel["age"] = panel["year"] - panel["year_of_incorporation"]
# Negative age means this firm-year row is for a year BEFORE the firm was incorporated
# -- an unbalanced panel (firms enter at different dates), not a data error. Drop those rows.
n_before_incorporation = (panel["age"] < 0).sum()
panel = panel.loc[panel["age"] >= 0].copy()
print(f"Dropped {n_before_incorporation:,} firm-year rows predating incorporation "
      f"({n_before_incorporation / (n_before_incorporation + len(panel)):.1%} of the stacked panel)")
panel["log_age"] = np.log1p(panel["age"])

# Selection: growth is observed only if the firm survives to t+1 AND both employment
# figures are actually reported. As section 4 will show, the second condition does
# almost all of the work here.
panel["selected"] = (
    panel["failure_t1"].eq(0)
    & panel["employees_t"].gt(0) & panel["employees_t"].notna()
    & panel["employees_t1"].gt(0) & panel["employees_t1"].notna()
).astype(int)
panel["growth"] = np.where(
    panel["selected"].eq(1),
    np.log(panel["employees_t1"]) - np.log(panel["employees_t"]),
    np.nan
)

print("Stacked panel shape (firm x year-pair rows):", panel.shape)
print("Year pairs included:", sorted(panel["year"].unique()))
print(f"Share of rows with observed growth (selected=1): {panel['selected'].mean():.1%}")

Dropped 349,742 firm-year rows predating incorporation (25.9% of the stacked panel)
Stacked panel shape (firm x year-pair rows): (1001610, 22)
Year pairs included: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015)]
Share of rows with observed growth (selected=1): 61.6%


## 4. What "selection" really means here

Only a minority of firm-year pairs have `failure_t1 = 1` -- exit in the classic sense is rare year to year. Run the cell below and compare that rate to the *overall* selection rate (which also requires employment to be reported in both years). The gap between the two numbers tells you something important: most of the attrition from this growth panel isn't firms failing, it's firms **not reporting employment data** at all.

This is exactly the informative-missingness story from the companion notebook, now hitting the *dependent variable* instead of a predictor. It has a real consequence for what comes next: Probit and OLS can't use rows with missing covariates, so fitting the Heckman model below requires restricting to **complete cases** on the covariates -- and we already know from the companion notebook that completeness of financial reporting is itself highly non-random. Check what that restriction does to the apparent selection rate; it is not innocuous.


In [5]:
fail_rate = panel["failure_t1"].mean()
selection_rate = panel["selected"].mean()
print(f"Share of firm-year pairs with failure_t1 = 1 : {fail_rate:.2%}")
print(f"Share of firm-year pairs with growth OBSERVED : {selection_rate:.2%}")
print(f"=> non-reporting (not failure) accounts for about "
      f"{(1 - selection_rate - fail_rate) / (1 - selection_rate):.0%} of all missing-growth rows")

# Exclusion-restriction ingredient + the covariates the selection/outcome equations need
grp = panel.groupby(["nace_2", "year"])["failure_t1"]
n, s = grp.transform("size"), grp.transform("sum")
panel["peer_exit_rate"] = (s - panel["failure_t1"]) / (n - 1)  # leave-one-out sector x year exit rate

MODEL_COVARS = ["tfp_acf", "liquidity_ratio", "solvency_ratio", "real_SA", "ICR_failure",
                "NEG_VA", "profitability", "fin_cons100", "log_age", "dummy_patents", "dummy_trademark"]
required_cols = MODEL_COVARS + ["peer_exit_rate", "nace_2", "year"]
complete_mask = panel[required_cols].notna().all(axis=1)

print(f"\nComplete-case rate on model covariates: {complete_mask.mean():.1%} of all firm-year pairs")
print(f"Selection rate WITHIN complete cases   : {panel.loc[complete_mask, 'selected'].mean():.1%}")
print("(compare to the overall selection rate above -- complete-case filtering has already")
print(" 'solved' most of the attrition problem by removing the firms most likely to attrit)")

model_panel = panel.loc[complete_mask].copy()


Share of firm-year pairs with failure_t1 = 1 : 0.98%
Share of firm-year pairs with growth OBSERVED : 61.58%
=> non-reporting (not failure) accounts for about 97% of all missing-growth rows

Complete-case rate on model covariates: 60.4% of all firm-year pairs
Selection rate WITHIN complete cases   : 92.3%
(compare to the overall selection rate above -- complete-case filtering has already
 'solved' most of the attrition problem by removing the firms most likely to attrit)


## 5. Selection equation

Formally, Heckman's model is identified even with identical regressors in both equations (the inverse Mills ratio is a nonlinear function of them), but it's best practice to include something in the selection equation that plausibly shifts survival without directly shifting the *growth rate* of firms that do survive. We use a **leave-one-out sector x year exit rate**: for each `(nace_2, year)` cell, the failure rate of *other* firms in the same sector and year (excluding the firm itself). The idea: an industry-wide downturn or shakeout raises any given firm's exit risk through channels like demand and credit conditions shared by the whole sector, but conditional on actually surviving that shakeout, a firm's own growth should mostly depend on its own characteristics rather than on how many neighbors happened to fail. This is a common instrument choice in the firm-dynamics literature (industry/local shakeout instruments) -- not a perfect exclusion restriction, but a defensible one, and worth stress-testing (see section 8).


In [6]:
sel_formula = ("selected ~ 0 + log_employees_t + log_age + tfp_acf + liquidity_ratio + solvency_ratio + "
               "real_SA + ICR_failure + NEG_VA + profitability + fin_cons100 + dummy_patents + dummy_trademark + "
               "peer_exit_rate + C(nace_2) + C(year)")

model_panel["log_employees_t"] = np.log(model_panel["employees_t"].clip(lower=0.5))

probit = smf.probit(sel_formula, data=model_panel).fit(disp=False, maxiter=100)
print(f"Selection equation: {len(model_panel):,} firm-year pairs, pseudo R-squared = {probit.prsquared:.3f}")
display(probit.summary2().tables[1].loc[
    ["log_employees_t", "log_age", "tfp_acf", "liquidity_ratio", "solvency_ratio", "real_SA",
     "ICR_failure", "NEG_VA", "profitability", "fin_cons100", "dummy_patents",
     "dummy_trademark", "peer_exit_rate"]
])

# Inverse Mills ratio, computed for every row (needed only for the selected ones next)
xb = probit.predict(model_panel, which="linear")
model_panel["imr"] = norm.pdf(xb) / norm.cdf(xb)


Selection equation: 604,567 firm-year pairs, pseudo R-squared = 0.227


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
log_employees_t,0.128244,0.006205,20.669052,6.579656e-95,0.116083,0.140404
log_age,0.006731,0.004703,1.431104,1.524004e-01,-0.002487,0.015949
tfp_acf,0.074192,0.006830,10.862599,1.737324e-27,0.060805,0.087578
liquidity_ratio,-0.013342,0.001166,-11.444637,2.501498e-30,-0.015627,-0.011057
solvency_ratio,0.000801,0.000145,5.536986,3.077211e-08,0.000517,0.001084
real_SA,-0.091158,0.005089,-17.912588,9.405907e-72,-0.101132,-0.081184
ICR_failure,-0.171302,0.006784,-25.249498,1.147024e-140,-0.184599,-0.158005
NEG_VA,-0.591137,0.017349,-34.073975,1.792394e-254,-0.625140,-0.557134
profitability,-0.090069,0.014554,-6.188542,6.072311e-10,-0.118594,-0.061543
fin_cons100,0.000138,0.000054,2.540105,1.108191e-02,0.000032,0.000245


## 6. Naive vs. Heckman-corrected growth regression

`naive` regresses growth on the same covariates using only the selected (surviving, reporting) firms -- the regression an analyst would run without thinking about selection at all. `heckman` adds the inverse Mills ratio computed above as an extra regressor. If its coefficient is statistically significant, that is direct evidence that the two equations' errors are correlated (formally, it estimates `rho * sigma`, where `rho` is that correlation) -- i.e. that the naive regression *was* biased, and by how the other coefficients shift once corrected.


In [7]:
outcome_rhs = ("log_employees_t + log_age + tfp_acf + liquidity_ratio + solvency_ratio + "
               "real_SA + ICR_failure + NEG_VA + profitability + fin_cons100 + dummy_patents + "
               "dummy_trademark + C(nace_2) + C(year)")

grown = model_panel.loc[model_panel["selected"].eq(1)].copy()
print(f"Outcome-equation sample: {len(grown):,} firm-year pairs with observed growth")

cluster_kwds = {"groups": grown["id"]}
naive = smf.ols(f"growth ~ {outcome_rhs}", data=grown).fit(cov_type="cluster", cov_kwds=cluster_kwds)
heckman = smf.ols(f"growth ~ {outcome_rhs} + imr", data=grown).fit(cov_type="cluster", cov_kwds=cluster_kwds)

KEY_VARS = ["log_employees_t", "log_age", "tfp_acf", "liquidity_ratio", "solvency_ratio", "real_SA",
            "ICR_failure", "NEG_VA", "profitability", "fin_cons100", "dummy_patents", "dummy_trademark"]

comparison = pd.DataFrame({
    "naive_coef": naive.params[KEY_VARS], "naive_p": naive.pvalues[KEY_VARS],
    "heckman_coef": heckman.params[KEY_VARS], "heckman_p": heckman.pvalues[KEY_VARS],
})
comparison["pct_change"] = 100 * (comparison["heckman_coef"] - comparison["naive_coef"]) / comparison["naive_coef"].abs()
display(comparison.style.format({
    "naive_coef": "{:.4f}", "naive_p": "{:.3f}", "heckman_coef": "{:.4f}",
    "heckman_p": "{:.3f}", "pct_change": "{:+.1f}%"
}))

print(f"\nInverse Mills ratio coefficient: {heckman.params['imr']:.4f} "
      f"(p = {heckman.pvalues['imr']:.4g})")
print("A significant, positive IMR coefficient means firms that survived despite a LOWER predicted")
print("survival probability tend to grow FASTER than observables alone would suggest -- i.e. the same")
print("unobserved shock that helps a firm survive against the odds also tends to help it grow.")


Outcome-equation sample: 557,990 firm-year pairs with observed growth


,naive_coef,naive_p,heckman_coef,heckman_p,pct_change
log_employees_t,-0.1410,0.000,-0.1355,0.000,+3.9%
log_age,-0.0668,0.000,-0.0660,0.000,+1.1%
tfp_acf,0.0545,0.000,0.0555,0.000,+1.8%
liquidity_ratio,0.0009,0.039,0.0002,0.669,-79.1%
solvency_ratio,-0.0004,0.000,-0.0004,0.000,+6.0%
real_SA,-0.0632,0.000,-0.0660,0.000,-4.5%
ICR_failure,-0.0569,0.000,-0.0630,0.000,-10.7%
NEG_VA,-0.0692,0.000,-0.1061,0.000,-53.4%
profitability,-0.0534,0.000,-0.0585,0.000,-9.4%
fin_cons100,-0.0001,0.000,-0.0001,0.000,+8.4%



Inverse Mills ratio coefficient: 0.1949 (p = 1.745e-87)
A significant, positive IMR coefficient means firms that survived despite a LOWER predicted
survival probability tend to grow FASTER than observables alone would suggest -- i.e. the same
unobserved shock that helps a firm survive against the odds also tends to help it grow.


### 6.1 A non-linear, missing-aware selection stage (Das-Newey-Vella style)

The Probit above needed complete cases: fitting it dropped every firm-year pair with any missing covariate, and section 4 already showed that this is not innocuous -- it disproportionately keeps the firms most likely to also report growth. **XGBoost handles missing values natively**, so we can fit the selection model on the *entire* stacked panel (all ~1,000,000 firm-year pairs, not just the ~600,000 complete cases), then bring the resulting selection probability into the outcome equation as a flexible correction term instead of a Probit-derived Mills ratio.

Two ways to do that, following the semiparametric sample-selection literature (Ahn & Powell, 1993; Das, Newey & Vella, 2003): (a) a "plug-in" Mills ratio, `mills_xgb = phi(Phi^-1(p_hat)) / p_hat`, which still assumes the normal-CDF link at the end; (b) a fully flexible control function -- here, a cubic polynomial in `p_hat` -- that doesn't assume the Mills-ratio shape is correct at all. We use out-of-fold predictions (`cross_val_predict`) for `p_hat` so the correction term isn't contaminated by the same overfitting that would result from evaluating XGBoost on the rows it was trained on.


In [8]:
def dense_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


NUM_COVARS = ["log_employees_t", "log_age", "tfp_acf", "liquidity_ratio", "solvency_ratio", "real_SA",
              "ICR_failure", "NEG_VA", "profitability", "fin_cons100", "dummy_patents", "dummy_trademark",
              "peer_exit_rate"]
CAT_COVARS = ["nace_2", "year"]

xgb_pre = ColumnTransformer([
    ("num", "passthrough", NUM_COVARS),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=-999)),
                       ("ohe", dense_ohe())]), CAT_COVARS)
])
xgb_selection = Pipeline([
    ("pre", xgb_pre),
    ("model", XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
        min_child_weight=5, reg_lambda=2.0, objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1
    ))
])

# Fit on the FULL panel (before the complete-case restriction) -- this is the whole point.
panel["log_employees_t"] = np.log(panel["employees_t"].clip(lower=0.5))
X_full = panel[NUM_COVARS + CAT_COVARS]
y_full = panel["selected"].to_numpy()
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Fitting XGBoost selection model on the full panel with 5-fold out-of-fold prediction "
      "(a few minutes)...")
p_oof = cross_val_predict(xgb_selection, X_full, y_full, cv=cv5, method="predict_proba", n_jobs=1)[:, 1]
panel["p_selected_xgb"] = np.clip(p_oof, 1e-4, 1 - 1e-4)
v_xgb = norm.ppf(panel["p_selected_xgb"])
panel["mills_xgb"] = norm.pdf(v_xgb) / panel["p_selected_xgb"]

print("Calibration check -- mean predicted P(selected):", panel["p_selected_xgb"].mean(),
      "| actual selection rate:", panel["selected"].mean())

# Bring the new correction terms onto the same complete-case / survivors sample used in section 6
model_panel = model_panel.merge(
    panel[["id", "year", "p_selected_xgb", "mills_xgb"]], on=["id", "year"], how="left"
)
grown = model_panel.loc[model_panel["selected"].eq(1)].copy()
grown["p2"], grown["p3"] = grown["p_selected_xgb"] ** 2, grown["p_selected_xgb"] ** 3

heckman_xgb_mills = smf.ols(f"growth ~ {outcome_rhs} + mills_xgb", data=grown).fit(
    cov_type="cluster", cov_kwds={"groups": grown["id"]})
heckman_xgb_poly = smf.ols(f"growth ~ {outcome_rhs} + p_selected_xgb + p2 + p3", data=grown).fit(
    cov_type="cluster", cov_kwds={"groups": grown["id"]})

four_way = pd.DataFrame({
    "naive": naive.params[KEY_VARS],
    "heckman_probit": heckman.params[KEY_VARS],
    "heckman_xgb_mills": heckman_xgb_mills.params[KEY_VARS],
    "heckman_xgb_poly": heckman_xgb_poly.params[KEY_VARS],
})
display(four_way.style.format("{:.4f}"))

print("\nCorrection-term coefficients (all p < 0.0001):")
print(f"  mills_probit (Probit, complete-case only) : {heckman.params['imr']:.4f}")
print(f"  mills_xgb (XGBoost, full panel)            : {heckman_xgb_mills.params['mills_xgb']:.4f}")
print(f"  correlation(mills_probit, mills_xgb) on survivors: {grown[['imr', 'mills_xgb']].corr().iloc[0, 1]:.3f}")


Fitting XGBoost selection model on the full panel with 5-fold out-of-fold prediction (a few minutes)...
Calibration check -- mean predicted P(selected): 0.61587924 | actual selection rate: 0.6158444903705035


,naive,heckman_probit,heckman_xgb_mills,heckman_xgb_poly
log_employees_t,-0.1410,-0.1355,-0.1389,-0.1364
log_age,-0.0668,-0.0660,-0.0667,-0.0661
tfp_acf,0.0545,0.0555,0.0541,0.0538
liquidity_ratio,0.0009,0.0002,0.0008,0.0005
solvency_ratio,-0.0004,-0.0004,-0.0004,-0.0004
real_SA,-0.0632,-0.0660,-0.0640,-0.0644
ICR_failure,-0.0569,-0.0630,-0.0585,-0.0612
NEG_VA,-0.0692,-0.1061,-0.0777,-0.0882
profitability,-0.0534,-0.0585,-0.0545,-0.0567
fin_cons100,-0.0001,-0.0001,-0.0001,-0.0001



Correction-term coefficients (all p < 0.0001):
  mills_probit (Probit, complete-case only) : 0.1949
  mills_xgb (XGBoost, full panel)            : 0.0541
  correlation(mills_probit, mills_xgb) on survivors: 0.956


### 6.2 Do we still need the exclusion restriction?

With a flexible, non-parametric first stage, does `peer_exit_rate` still matter, or can XGBoost's non-linear combination of the firm's own characteristics substitute for it? In principle dropping it should be riskier here, not safer: if the correction term is some flexible function of the *same* covariates used in the outcome equation, separating "the direct effect of X on growth" from "the correction for selection on X" leans entirely on functional form -- an even sharper version of the classic identification-by-functional-form critique of Heckman's model (Puhani, 2000).

The cell below refits the XGBoost selection model without `peer_exit_rate` and checks two things: (1) how much of the resulting correction term is just a (possibly nonlinear) function of the outcome equation's own covariates -- via the R^2 of regressing the correction term on them -- and (2) whether the key growth coefficients actually move.


In [9]:
NUM_COVARS_NOINSTR = [c for c in NUM_COVARS if c != "peer_exit_rate"]
xgb_pre_noinstr = ColumnTransformer([
    ("num", "passthrough", NUM_COVARS_NOINSTR),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=-999)),
                       ("ohe", dense_ohe())]), CAT_COVARS)
])
xgb_selection_noinstr = Pipeline([("pre", xgb_pre_noinstr), ("model", XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
    min_child_weight=5, reg_lambda=2.0, objective="binary:logistic", eval_metric="logloss",
    tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1))])

p_oof_noinstr = cross_val_predict(
    xgb_selection_noinstr, panel[NUM_COVARS_NOINSTR + CAT_COVARS], y_full, cv=cv5,
    method="predict_proba", n_jobs=1
)[:, 1]
panel["p_selected_xgb_noinstr"] = np.clip(p_oof_noinstr, 1e-4, 1 - 1e-4)
v_noinstr = norm.ppf(panel["p_selected_xgb_noinstr"])
panel["mills_xgb_noinstr"] = norm.pdf(v_noinstr) / panel["p_selected_xgb_noinstr"]

grown = grown.merge(panel[["id", "year", "mills_xgb_noinstr"]], on=["id", "year"], how="left")

r2_with = smf.ols(f"mills_xgb ~ {outcome_rhs}", data=grown).fit().rsquared
r2_without = smf.ols(f"mills_xgb_noinstr ~ {outcome_rhs}", data=grown).fit().rsquared
print(f"R^2 of correction term ~ outcome covariates,    WITH peer_exit_rate: {r2_with:.4f}")
print(f"R^2 of correction term ~ outcome covariates, WITHOUT peer_exit_rate: {r2_without:.4f}")

heckman_noinstr = smf.ols(f"growth ~ {outcome_rhs} + mills_xgb_noinstr", data=grown).fit(
    cov_type="cluster", cov_kwds={"groups": grown["id"]})

instrument_check = pd.DataFrame({
    "with_instrument": heckman_xgb_mills.params[KEY_VARS],
    "without_instrument": heckman_noinstr.params[KEY_VARS],
})
display(instrument_check.style.format("{:.4f}"))
print(f"\nmills_xgb coefficient (with instrument)   : {heckman_xgb_mills.params['mills_xgb']:.4f}")
print(f"mills_xgb_noinstr coefficient (without)    : {heckman_noinstr.params['mills_xgb_noinstr']:.4f}")
print("\nIf the R^2 values and the coefficient table above are both similar with and without the")
print("instrument, the non-linear first stage is doing enough of the identifying work on its own")
print("in this particular application -- but that's an empirical finding here, not a general license")
print("to skip exclusion restrictions in semiparametric selection models elsewhere.")


R^2 of correction term ~ outcome covariates,    WITH peer_exit_rate: 0.7804
R^2 of correction term ~ outcome covariates, WITHOUT peer_exit_rate: 0.7812


,with_instrument,without_instrument
log_employees_t,-0.1389,-0.1387
log_age,-0.0667,-0.0667
tfp_acf,0.0541,0.0541
liquidity_ratio,0.0008,0.0007
solvency_ratio,-0.0004,-0.0004
real_SA,-0.0640,-0.0642
ICR_failure,-0.0585,-0.0586
NEG_VA,-0.0777,-0.0792
profitability,-0.0545,-0.0547
fin_cons100,-0.0001,-0.0001



mills_xgb coefficient (with instrument)   : 0.0541
mills_xgb_noinstr coefficient (without)    : 0.0613

If the R^2 values and the coefficient table above are both similar with and without the
instrument, the non-linear first stage is doing enough of the identifying work on its own
in this particular application -- but that's an empirical finding here, not a general license
to skip exclusion restrictions in semiparametric selection models elsewhere.


### 6.3 Firm-level heterogeneity: is the pooled model hiding a within/between split?

Every regression so far pools all firm-year pairs together and controls only for sector and year -- it says nothing about unobserved, time-invariant firm characteristics (management quality, an unmeasured cost advantage, ...) that could be correlated with both a firm's covariates and its growth. The natural fix, firm fixed effects, runs into two specific obstacles here:

1. **Age-period-cohort collinearity.** `age_it = t - year_of_incorporation_i`. With full firm fixed effects (which absorb `year_of_incorporation_i`) *and* full year fixed effects, `age` becomes an exact linear combination of the two (`age_it = t + (-year_of_incorporation_i)`) -- it stops being separately identified. This is the classic age-period-cohort problem from demography and labor economics.
2. **Incidental parameters in the Probit.** Firm fixed effects in a Probit with ~169,000 firms and at most 8 periods each are inconsistent (the Neyman-Scott problem) -- unlike linear fixed effects, which are unbiased regardless of how many periods you observe per firm.

**Mundlak's (1978) device** sidesteps both: instead of firm dummies, add each firm's *own average* of every time-varying regressor (across all years it appears in the panel) as an extra control, in both equations. Firms with permanently high productivity, permanently tight financial constraints, etc. now have that captured by the mean terms, while the coefficient on the *level* of each variable becomes a "within" estimate -- the effect of a firm deviating from its own average, which is what firm fixed effects would give you (Chamberlain, 1980, shows the two coincide under Mundlak's linearity assumption). Because it only adds a handful of extra continuous regressors rather than thousands of dummies, it avoids the incidental-parameters problem, and because it doesn't touch `age` with a full dummy set, it avoids the collinearity problem too.


**A sanity check worth knowing about**: the `log_employees_t` "within" coefficient typically comes out *far* more negative under Mundlak than in the pooled model -- often several times larger in magnitude. That is not a bug. Firms that are *persistently* larger tend to have somewhat higher long-run growth (a positive "between" effect -- plausibly reflecting scale advantages or a survivorship-selected set of successful large firms), while a given firm growing unusually large *relative to its own history* tends to see growth slow sharply afterward (a strongly negative "within" effect, i.e. an even stronger rejection of Gibrat's law than section 6 suggested). These two effects partly cancel in the pooled coefficient, which is exactly the kind of thing Mundlak's decomposition is designed to reveal. You can verify this isn't a modeling artifact by comparing `mundlak_within` above to a true within-firm (demeaned) OLS regression restricted to firms observed in 2+ periods -- the two match almost exactly, because that's what Mundlak's device is built to reproduce for a linear outcome equation.


In [10]:
MEAN_VARS = ["log_employees_t", "log_age", "tfp_acf", "liquidity_ratio", "solvency_ratio",
             "real_SA", "ICR_failure", "NEG_VA", "profitability", "fin_cons100"]

firm_means = panel.groupby("id")[MEAN_VARS].mean().add_suffix("_mean")
panel = panel.merge(firm_means, on="id", how="left")
mundlak_terms = " + ".join(f"{v}_mean" for v in MEAN_VARS)

model_panel = model_panel.merge(panel[["id", "year"] + [f"{v}_mean" for v in MEAN_VARS]], on=["id", "year"], how="left")

probit_mundlak = smf.probit(f"selected ~ 0 + {sel_formula.split('~ 0 + ')[1]} + {mundlak_terms}",
                             data=model_panel).fit(disp=False, maxiter=100)
xb_mundlak = probit_mundlak.predict(model_panel, which="linear")
model_panel["mills_mundlak"] = norm.pdf(xb_mundlak) / norm.cdf(xb_mundlak)

grown = model_panel.loc[model_panel["selected"].eq(1)].copy()
heckman_mundlak = smf.ols(f"growth ~ {outcome_rhs} + {mundlak_terms} + mills_mundlak", data=grown).fit(
    cov_type="cluster", cov_kwds={"groups": grown["id"]})

within_comparison = pd.DataFrame({
    "pooled_heckman": heckman.params[KEY_VARS],
    "mundlak_within": heckman_mundlak.params[KEY_VARS],
})
display(within_comparison.style.format("{:.4f}"))

print("\nBetween-firm (mean) terms -- the permanent, cross-firm relationship:")
display(heckman_mundlak.params[[f"{v}_mean" for v in MEAN_VARS]].to_frame("between_coef"))

ftest = heckman_mundlak.f_test(" = 0, ".join(f"{v}_mean" for v in MEAN_VARS) + " = 0")
print(f"\nJoint significance of all firm-mean terms: F = {float(ftest.fvalue):.1f}, p = {float(ftest.pvalue):.4g}")
print("A large, significant F here means firm-level permanent heterogeneity really is confounding")
print("the pooled estimates above -- not just a theoretical possibility.")


,pooled_heckman,mundlak_within
log_employees_t,-0.1355,-0.6717
log_age,-0.0660,-0.0251
tfp_acf,0.0555,0.0788
liquidity_ratio,0.0002,-0.0012
solvency_ratio,-0.0004,0.0011
real_SA,-0.0660,-0.1700
ICR_failure,-0.0630,-0.0284
NEG_VA,-0.1061,-0.0532
profitability,-0.0585,-0.0251
fin_cons100,-0.0001,0.0000



Between-firm (mean) terms -- the permanent, cross-firm relationship:


,between_coef
log_employees_t_mean,0.648954
log_age_mean,-0.006979
tfp_acf_mean,-0.072224
liquidity_ratio_mean,-0.000133
solvency_ratio_mean,-0.001346
real_SA_mean,0.154450
ICR_failure_mean,-0.046351
NEG_VA_mean,0.026413
profitability_mean,-0.020935
fin_cons100_mean,-0.000222



Joint significance of all firm-mean terms: F = 2820.3, p = 0
A large, significant F here means firm-level permanent heterogeneity really is confounding
the pooled estimates above -- not just a theoretical possibility.


### 6.4 Measurement error and division bias

`growth = log(employees_{t+1}) - log(employees_t)`, and `log(employees_t)` is *also* a right-hand-side regressor (initial size). If `employees_t` is measured with error, that error enters the model **twice with opposite signs**: an upward error makes the firm look bigger (in the regressor) *and* makes measured growth look smaller (it inflates the value being subtracted). This mechanically manufactures a negative size-growth relationship regardless of any true economic effect -- the classic critique of naive Gibrat's-law tests (Chesher, 1979; Friedman, 1992; surveyed in Sutton, 1997).

It is not just a theoretical worry here: section 6.3 showed employment is reported with a suspiciously high year-to-year "no change at all" rate for small firms, consistent with some values being carried forward rather than freshly measured.

**The standard fix**: instrument `log(employees_t)` with `log(employees_{t-1})`. If the measurement error is not strongly serially correlated, a lagged value is correlated with true size but not with *this period's* specific error -- exactly the logic of an errors-in-variables IV correction. Combining this with the selection correction means running **2SLS instead of OLS** in the outcome equation, while still including a Mills-ratio-style correction term as a plain (already-estimated) exogenous regressor -- this combination is discussed in Wooldridge (2010) and, for the panel case, in Vella & Verbeek (1999).


In [11]:
# Rebuild the panel starting one year later so employees_{t-1} exists as an instrument.
panel_iv = []
for t in range(2009, 2016):
    emp_tm1, emp_t, emp_t1 = year_col("employees", t - 1), year_col("employees", t), year_col("employees", t + 1)
    fail_t1 = f"failure_{t + 1}"
    cols = [emp_tm1, emp_t, emp_t1, fail_t1] + [year_col(b, t) for b in COVARIATE_BASES]
    sub = wide[["id", "nace_2", "region", "dummy_patents", "dummy_trademark",
                "year_of_incorporation"] + cols].copy()
    sub = sub.rename(columns={emp_tm1: "employees_tm1", emp_t: "employees_t",
                               emp_t1: "employees_t1", fail_t1: "failure_t1"})
    sub = sub.rename(columns={year_col(b, t): b for b in COVARIATE_BASES})
    sub["year"] = t
    panel_iv.append(sub)
panel_iv = pd.concat(panel_iv, ignore_index=True)
panel_iv["age"] = panel_iv["year"] - panel_iv["year_of_incorporation"]
panel_iv = panel_iv.loc[panel_iv["age"] >= 0].copy()
panel_iv["log_age"] = np.log1p(panel_iv["age"])
panel_iv["log_employees_t"] = np.log(panel_iv["employees_t"].clip(lower=0.5))

panel_iv["selected"] = (
    panel_iv["failure_t1"].eq(0) & panel_iv["employees_t"].gt(0) & panel_iv["employees_t"].notna()
    & panel_iv["employees_t1"].gt(0) & panel_iv["employees_t1"].notna()
    & panel_iv["employees_tm1"].gt(0) & panel_iv["employees_tm1"].notna()
).astype(int)
panel_iv["growth"] = np.where(panel_iv["selected"].eq(1),
    np.log(panel_iv["employees_t1"]) - np.log(panel_iv["employees_t"]), np.nan)

required_iv = MODEL_COVARS + ["nace_2", "year", "log_employees_t", "employees_tm1"]
cc_iv = panel_iv.loc[panel_iv[required_iv].notna().all(axis=1)].copy()
cc_iv["nace_2_int"] = cc_iv["nace_2"].astype(int)
grown_iv = cc_iv.loc[cc_iv["selected"].eq(1)].copy()
grown_iv["log_employees_tm1"] = np.log(grown_iv["employees_tm1"].clip(lower=0.5))
print("IV sample (2009-2015 pairs, employees_t-1 required):", len(grown_iv))

year_dum = pd.get_dummies(grown_iv["year"], prefix="y", drop_first=True).astype(float)
sector_dum = pd.get_dummies(grown_iv["nace_2_int"], prefix="s", drop_first=True).astype(float)
grown_iv = pd.concat([grown_iv.reset_index(drop=True), year_dum.reset_index(drop=True),
                       sector_dum.reset_index(drop=True)], axis=1)
exog_iv = ["log_age", "tfp_acf", "liquidity_ratio", "solvency_ratio", "real_SA", "ICR_failure",
           "NEG_VA", "profitability", "fin_cons100", "dummy_patents", "dummy_trademark"] \
          + list(year_dum.columns) + list(sector_dum.columns)

naive_iv_sample = smf.ols("growth ~ log_employees_t + " + " + ".join(exog_iv), data=grown_iv).fit(
    cov_type="cluster", cov_kwds={"groups": grown_iv["id"]})

iv_only = IV2SLS.from_formula(
    "growth ~ 1 + " + " + ".join(exog_iv) + " + [log_employees_t ~ log_employees_tm1]", data=grown_iv
).fit(cov_type="clustered", clusters=grown_iv["id"])

first_stage = smf.ols("log_employees_t ~ log_employees_tm1 + " + " + ".join(exog_iv), data=grown_iv).fit()
print(f"First-stage F-stat on the instrument: {first_stage.tvalues['log_employees_tm1']**2:.0f} "
      "(>> 10 means not a weak instrument)")

print("\nlog_employees_t coefficient:")
print(f"  naive OLS (same sample) : {naive_iv_sample.params['log_employees_t']:.4f}")
print(f"  2SLS (IV-corrected)     : {iv_only.params['log_employees_t']:.4f}")
print(f"\nShrinkage from correcting division bias: "
      f"{100 * (1 - iv_only.params['log_employees_t'] / naive_iv_sample.params['log_employees_t']):.0f}%")
print("\nCompare to section 6's headline naive/Heckman estimate (~-0.135): division bias, not")
print("selection bias, turns out to explain most of the apparent gap from zero for this coefficient.")


IV sample (2009-2015 pairs, employees_t-1 required): 438459
First-stage F-stat on the instrument: 444908 (>> 10 means not a weak instrument)

log_employees_t coefficient:
  naive OLS (same sample) : -0.1262
  2SLS (IV-corrected)     : -0.0447

Shrinkage from correcting division bias: 65%

Compare to section 6's headline naive/Heckman estimate (~-0.135): division bias, not
selection bias, turns out to explain most of the apparent gap from zero for this coefficient.


## 7. A note on standard errors

The cluster-robust standard errors above (clustered by firm `id`, since the same firm contributes several year-pairs) address one real problem -- within-firm serial correlation -- but not the classic **Heckman two-step generated-regressor problem**: the inverse Mills ratio `imr` is itself an *estimate* from the first-stage Probit, not an observed variable, and plain OLS standard errors (even clustered ones) don't know that. The standard fix is either (a) the analytical variance correction from Heckman (1979), or (b) a **block bootstrap**: resample firms (not firm-years) with replacement, rerun the whole two-step procedure on each resample, and use the spread of the resulting coefficients as the standard error.

We don't run the bootstrap by default here -- with ~250,000 firms and a Probit plus OLS per replication, even 200 replications would take a while for a first pass. If you need publication-quality inference, that is the next thing to add; a starting point is sketched below (`N_BOOTSTRAP = 0` skips it).


In [12]:
N_BOOTSTRAP = 0  # set e.g. to 200 for a real bootstrap; expect a long runtime

def one_bootstrap_rep(panel_df, seed):
    firms = panel_df["id"].unique()
    rng = np.random.default_rng(seed)
    sampled_firms = rng.choice(firms, size=len(firms), replace=True)
    boot = panel_df.set_index("id").loc[sampled_firms].reset_index()
    p = smf.probit(sel_formula, data=boot).fit(disp=False, maxiter=50)
    xb_b = p.predict(boot, which="linear")
    boot["imr"] = norm.pdf(xb_b) / norm.cdf(xb_b)
    g = boot.loc[boot["selected"].eq(1)]
    m = smf.ols(f"growth ~ {outcome_rhs} + imr", data=g).fit()
    return m.params[KEY_VARS + ["imr"]]

if N_BOOTSTRAP > 0:
    boot_coefs = pd.concat(
        [one_bootstrap_rep(model_panel, seed) for seed in range(N_BOOTSTRAP)], axis=1
    ).T
    display(boot_coefs.agg(["mean", "std"]).T.rename(columns={"std": "bootstrap_se"}))
else:
    print("N_BOOTSTRAP = 0: skipped. Set it above and rerun this cell for bootstrap standard errors.")


N_BOOTSTRAP = 0: skipped. Set it above and rerun this cell for bootstrap standard errors.


## 8. Limitations and extensions

1. **The complete-case restriction still binds for the outcome equation.** Section 6.1's XGBoost selection stage fixes this for the *selection* equation (it uses the full ~1,000,000-row panel), but the second-stage OLS/2SLS still needs complete cases for its own covariates.
2. **The exclusion restriction turned out not to matter much here (section 6.2), but that's an empirical finding for this dataset, not a general result.**
3. **Two-step vs. full maximum likelihood.** We used the classic Heckman two-step estimator for transparency; a full-information MLE version (jointly estimating both equations) is more efficient.
4. **The semiparametric approach in 6.1 follows Ahn & Powell (1993) and Das, Newey & Vella (2003)**; cross-fitting it (out-of-fold predictions) connects to "double/debiased machine learning" (Chernozhukov et al., 2018).
5. **Division bias (section 6.4) and firm heterogeneity (section 6.3) pull the size coefficient in opposite directions and were corrected separately, not jointly.** The Mundlak-within estimate (-0.672) purges permanent firm heterogeneity but still uses the mismeasured `log_employees_t` directly; the 2SLS estimate (-0.040-0.045) purges the measurement error but is a pooled (not within-firm) estimate. Combining both -- an IV estimated *within* a Mundlak/fixed-effects specification -- is exactly what section 11's final exercise asks you to attempt.
6. **Sales growth, other countries.** Section 10 asks you to redo the core comparison with `revenue` instead of `employees`; switching `COUNTRY` to `"FR"`, `"PT"`, or `"ES"` is mechanically simple but expect smaller samples and higher missingness (companion notebook, section 2).
7. **The standard-error bootstrap in section 7** is the piece most worth prioritizing next for any of these specifications if this analysis needs to support a real claim rather than a classroom illustration -- and it would need to cover the IV and Mundlak specifications too, not just the classic Heckman one.


## 9. Exporting the results


In [13]:
OUTPUT_DIR = Path("/content/growth_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

comparison.to_csv(OUTPUT_DIR / "naive_vs_heckman_coefficients.csv")
probit.summary2().tables[1].to_csv(OUTPUT_DIR / "selection_equation.csv")
model_panel.to_parquet(OUTPUT_DIR / "growth_panel.parquet", index=False)

print("Files created:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path)


Files created:
- /content/growth_outputs/growth_panel.parquet
- /content/growth_outputs/naive_vs_heckman_coefficients.csv
- /content/growth_outputs/selection_equation.csv


## 10. Homework exercise: does the story change for sales growth?

Everything above uses **employment** growth. Redo the analysis with **sales (revenue)** growth instead, and compare. To be handed in as a discussion (working code is a bonus, not a requirement):

1. Rebuild the panel from section 3, but track `revenue_t` / `revenue_t1` instead of `employees_t` / `employees_t1` (the `year_col()` helper and the covariate list are already generic -- you mostly just need to swap which pair of columns defines growth and selection).
2. Look at the **selection rate** before doing anything else: is revenue reported as completely as employment was? What does that suggest about which kind of missingness (failure vs. non-disclosure) dominates for this outcome?
3. Look at the **distribution of the growth measure** before running any regression. Employment growth in section 3 had a standard deviation around 0.35; check what you get for sales growth, and decide whether winsorizing extreme values is warranted here.
4. Run the same naive-vs-Heckman comparison as section 6. Pay particular attention to:
   - the coefficient on `log_employees_t` (initial size) -- does smaller-grows-faster still hold, or could it reverse for a different growth margin?
   - whether correcting for selection ever *flips the sign* of a coefficient, not just its magnitude -- and if so, what that would have meant for a policy conclusion based on the naive regression alone.
5. If you have time, redo the Mundlak decomposition from section 6.3 for sales growth and compare the within/between split to the employment version. A growth measure that moves quickly and is more prone to mean-reversion (a bad year often followed by a rebound) can behave quite differently within a firm than one that adjusts slowly (like employment, given hiring/firing frictions) -- see if that shows up in your results.
6. Discuss: which growth measure -- employment or sales -- gives the more economically meaningful picture of "growth" for the questions this notebook is trying to answer, and why might a researcher want to look at both rather than picking one?


## 11. Final homework: combining all three corrections

Sections 6.3 and 6.4 corrected the `log_employees_t` coefficient for two *separate* problems, in opposite directions:

| Correction | What it fixes | Effect on the coefficient |
|---|---|---|
| None (pooled/naive) | -- | $-0.126$ |
| Mundlak within (6.3) | permanent firm heterogeneity | $-0.672$ (much *more* negative) |
| 2SLS (6.4) | division bias / measurement error | $-0.040$ (much *less* negative) |

These were never applied together. The exercise: build a specification that corrects for **both** at once, plus the selection correction from section 6.1.

Suggested approach (this is a genuinely open-ended exercise -- there is more than one defensible way to do it):

1. Add the firm-level means from section 6.3 (`*_mean` columns) as controls in the **2SLS** outcome equation from section 6.4, the same way they were added to the plain OLS outcome equation in 6.3. You will need an instrument for `log_employees_t` that is still valid conditional on the firm mean of `log_employees_t` -- is `log_employees_{t-1}` still usable, or does it need its own Mundlak-style mean term added as well?
2. Re-estimate the selection equation (Probit or XGBoost) including the same Mundlak mean terms, to keep the selection-correction step consistent with the rest of the specification.
3. Compare the resulting `log_employees_t` coefficient to all three estimates in the table above. Is it closer to $-0.67$, to $-0.04$, or somewhere else entirely? What does that tell you about which bias was actually larger in this data?
4. `linearmodels.iv.IV2SLS` accepts additional exogenous regressors directly in its formula (as sections 6.4's code already shows) -- extending it with more controls is mechanical. Getting valid standard errors for the fully combined estimator is not: at that point you are firmly in "this needs the bootstrap from section 7" territory.


### 11.1 Implementation of steps 1-2

Everything below runs on the section 6.4 IV sample (2009-2015 pairs, `employees_{t-1}` required), so all specifications in the final table share **one** estimation sample.

**Step 1: which mean terms go into the 2SLS equation?** Adding all section 6.3 means literally, including `log_employees_t_mean`, causes a problem. That mean averages `log(employees)` over the firm's years, *including year t*, so it contains a `1/T_i` share of the same measurement error `e_t` that the instrument is meant to remove. Treating it as exogenous brings back a diluted form of division bias. The correlated-random-effects version of 2SLS (Wooldridge, 2010, ch. 11; Wooldridge, 2019) instead includes the firm means of the **exogenous variables and the instruments**, never of the endogenous regressor. So the preferred specification replaces `log_employees_t_mean` with `log_employees_tm1_mean` (the firm mean of the instrument), and `log_employees_{t-1}` stays the excluded instrument. The literal version is also estimated, for comparison.

**Step 2: selection equation with Mundlak means.** A pooled Probit on the IV complete-case panel with `peer_exit_rate` as the exclusion restriction, plus the same mean terms as the matching outcome equation (Semykina & Wooldridge, 2010, is the reference for panel selection + endogeneity + Mundlak means). In the preferred version the Probit uses `log_employees_tm1` rather than `log_employees_t`. If the Mills ratio were a function of the mismeasured current size, it would carry `e_t` into the 2SLS equation as a regressor that is supposed to be exogenous. Note also that section 6.4 had **no** selection term at all, so specs (5) vs (6) isolate what the selection correction adds on top of IV + Mundlak.

**Caveats to discuss:** (i) `log_employees_{t-1}` is only *sequentially* exogenous. Its firm mean includes `log_employees_t` from the firm's other rows, so some `O(1/T)` contamination remains with `T <= 7`. The clean fix is first differences with deeper lags (Anderson-Hsiao / Arellano-Bond). (ii) Standard errors are firm-clustered but ignore the generated Mills regressor, so step 4 (bootstrap) still applies.

In [14]:
# ---- 11.1a  Common sample + Mundlak terms for the combined estimator -------------------------
MEANS_A = [f"{v}_mean" for v in MEAN_VARS]                               # literal: all 6.3 means
MEANS_B = [f"{v}_mean" for v in MEAN_VARS if v != "log_employees_t"] \
          + ["log_employees_tm1_mean"]                                  # CRE-2SLS: instrument mean instead

# Firm mean of the instrument over all rows of the IV panel where employees_{t-1} is reported
panel_iv["log_employees_tm1"] = np.log(panel_iv["employees_tm1"].clip(lower=0.5))
iv_means = panel_iv.groupby("id")[["log_employees_tm1"]].mean().add_suffix("_mean")

cc = (cc_iv
      .merge(firm_means, on="id", how="left")                            # section 6.3 means
      .merge(iv_means, on="id", how="left")
      .merge(panel[["id", "year", "peer_exit_rate"]], on=["id", "year"], how="left"))
cc["log_employees_tm1"] = np.log(cc["employees_tm1"].clip(lower=0.5))
n_before = len(cc)
cc = cc.dropna(subset=MEANS_A + ["log_employees_tm1_mean", "peer_exit_rate"]).reset_index(drop=True)
print(f"Selection-equation sample (IV complete cases): {len(cc):,} firm-year pairs "
      f"({n_before - len(cc):,} dropped for missing means / peer_exit_rate); "
      f"selection rate {cc['selected'].mean():.1%}")
print(f"corr(log_employees_t_mean, log_employees_tm1_mean) = "
      f"{cc[['log_employees_t_mean', 'log_employees_tm1_mean']].corr().iloc[0, 1]:.3f}"
      "  (why only one of the two can enter)")

Selection-equation sample (IV complete cases): 464,732 firm-year pairs (0 dropped for missing means / peer_exit_rate); selection rate 94.3%
corr(log_employees_t_mean, log_employees_tm1_mean) = 0.998  (why only one of the two can enter)


In [15]:
# ---- 11.1b  STEP 2: selection equation with Mundlak means -----------------------------------
SEL_BASE = ("log_age + tfp_acf + liquidity_ratio + solvency_ratio + real_SA + ICR_failure + NEG_VA + "
            "profitability + fin_cons100 + dummy_patents + dummy_trademark + peer_exit_rate + "
            "C(nace_2) + C(year)")

def mundlak_probit_mills(df, size_var, mean_terms):
    f = f"selected ~ 0 + {size_var} + {SEL_BASE} + " + " + ".join(mean_terms)
    fit = smf.probit(f, data=df).fit(disp=False, maxiter=100)
    xb = fit.predict(df, which="linear")
    return fit, np.exp(norm.logpdf(xb) - norm.logcdf(xb))   # numerically stable phi/Phi

# A: literal (current size + all 6.3 means)   B: preferred (lagged size + CRE means)
probit_A, cc["mills_A"] = mundlak_probit_mills(cc, "log_employees_t", MEANS_A)
probit_B, cc["mills_B"] = mundlak_probit_mills(cc, "log_employees_tm1", MEANS_B)

for name, fit, size_var, means in [("A (literal)", probit_A, "log_employees_t", MEANS_A),
                                   ("B (preferred)", probit_B, "log_employees_tm1", MEANS_B)]:
    print(f"\nProbit {name}: n = {int(fit.nobs):,}, pseudo R^2 = {fit.prsquared:.3f}, converged = "
          f"{fit.mle_retvals.get('converged')}")
    tab = fit.summary2().tables[1].loc[[size_var, "peer_exit_rate"] + means, ["Coef.", "Std.Err.", "P>|z|"]]
    display(tab.style.format({"Coef.": "{:.4f}", "Std.Err.": "{:.4f}", "P>|z|": "{:.3g}"}))
    wald = fit.wald_test(", ".join(f"{m} = 0" for m in means), scalar=True)
    print(f"Joint Wald test, all mean terms = 0: chi2 = {float(wald.statistic):.1f}, p = {float(wald.pvalue):.3g}")

print(f"\ncorr(mills_A, mills_B) on survivors: "
      f"{cc.loc[cc['selected'].eq(1), ['mills_A', 'mills_B']].corr().iloc[0, 1]:.3f}")

# Outcome-equation sample: survivors from the IV panel, with means and correction terms attached
grown_c = grown_iv.merge(
    cc[["id", "year"] + MEANS_A + ["log_employees_tm1_mean", "mills_A", "mills_B"]],
    on=["id", "year"], how="inner")
print(f"Outcome-equation sample: {len(grown_c):,} (section 6.4 had {len(grown_iv):,})")


Probit A (literal): n = 464,732, pseudo R^2 = 0.228, converged = True


,Coef.,Std.Err.,P>|z|
log_employees_t,0.2578,0.0159,3.14e-59
peer_exit_rate,0.8982,1.4278,0.529
log_employees_t_mean,-0.1485,0.0182,3.16e-16
log_age_mean,0.1881,0.0238,2.89e-15
tfp_acf_mean,-0.1817,0.0194,7.53e-21
liquidity_ratio_mean,-0.0143,0.0025,1.91e-08
solvency_ratio_mean,-0.0009,0.0005,0.0649
real_SA_mean,-0.1653,0.0165,1.26e-23
ICR_failure_mean,-0.2541,0.0178,1.96e-46
NEG_VA_mean,-0.2821,0.0466,1.41e-09


Joint Wald test, all mean terms = 0: chi2 = 1148.5, p = 1.85e-240

Probit B (preferred): n = 464,732, pseudo R^2 = 0.227, converged = True


,Coef.,Std.Err.,P>|z|
log_employees_tm1,0.0109,0.0107,0.309
peer_exit_rate,0.9951,1.4268,0.486
log_age_mean,0.1969,0.0238,1.28e-16
tfp_acf_mean,-0.3636,0.0150,1.2e-130
liquidity_ratio_mean,-0.0136,0.0025,8.95e-08
solvency_ratio_mean,-0.0010,0.0005,0.0323
real_SA_mean,-0.1614,0.0165,1.66e-22
ICR_failure_mean,-0.2585,0.0177,3.49e-48
NEG_VA_mean,-0.2752,0.0466,3.41e-09
profitability_mean,0.0432,0.0408,0.29


Joint Wald test, all mean terms = 0: chi2 = 1165.0, p = 5.05e-244

corr(mills_A, mills_B) on survivors: 0.996
Outcome-equation sample: 438,459 (section 6.4 had 438,459)


In [16]:
# ---- 11.1c-0  Free RAM before the 2SLS fits ---------------------------------------------------
# Everything 11.1c needs is in `grown_c` (+ exog_iv, MEANS_A/B, OUTPUT_DIR) and three reference
# coefficients. statsmodels/linearmodels results keep their full design matrices in memory
# (hundreds of MB each with ~100 sector/year dummies), so they are the main thing to drop.
import gc, ctypes, psutil

def ram():
    return f"{psutil.Process().memory_info().rss / 1e9:.2f} GB used by this notebook"

print("Before:", ram())

REF = {  # reference values from the section 11 table, kept as plain numbers
    "pooled Heckman (6)":   float(heckman.params["log_employees_t"]),
    "Mundlak within (6.3)": float(heckman_mundlak.params["log_employees_t"]),
    "2SLS (6.4)":           float(iv_only.params["log_employees_t"]),
}

# Slim the outcome sample down to the columns actually used below
KEEP = (["id", "growth", "log_employees_t", "log_employees_tm1"] + exog_iv
        + MEANS_A + ["log_employees_tm1_mean", "mills_A", "mills_B"])
grown_c = grown_c[list(dict.fromkeys(KEEP))].copy()
grown_c["const"] = 1.0

BIG = ["wide", "panel", "panels", "sub", "model_panel", "grown", "X_full", "y_full", "p_oof", "p_oof_noinstr",
       "v_xgb", "v_noinstr", "xb", "xb_mundlak", "xgb_selection", "xgb_selection_noinstr",
       "probit", "probit_mundlak", "probit_A", "probit_B", "fit", "tab", "wald",
       "naive", "heckman", "heckman_xgb_mills", "heckman_xgb_poly", "heckman_noinstr", "heckman_mundlak",
       "naive_iv_sample", "iv_only", "first_stage", "panel_iv", "cc_iv", "cc", "grown_iv",
       "firm_means", "iv_means", "year_dum", "sector_dum", "r", "combined_fits", "boot_coefs"]
dropped = [n for n in BIG if globals().pop(n, None) is not None]
print("Dropped:", ", ".join(dropped))

get_ipython().run_line_magic("reset", "-f out")   # IPython's Out[] cache can pin old objects
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)          # hand freed memory back to the OS (Linux/Colab)
except OSError:
    pass
print("After: ", ram(), f"| grown_c: {grown_c.memory_usage(deep=True).sum() / 1e9:.2f} GB, {len(grown_c):,} rows")

Before: 9.32 GB used by this notebook
Dropped: wide, panel, panels, sub, model_panel, grown, X_full, y_full, p_oof, p_oof_noinstr, v_xgb, v_noinstr, xb, xb_mundlak, xgb_selection, xgb_selection_noinstr, probit, probit_mundlak, probit_A, probit_B, fit, tab, wald, naive, heckman, heckman_xgb_mills, heckman_xgb_poly, heckman_noinstr, heckman_mundlak, naive_iv_sample, iv_only, first_stage, panel_iv, cc_iv, cc, grown_iv, firm_means, iv_means, year_dum, sector_dum
Flushing output cache (0 entries)
After:  0.92 GB used by this notebook | grown_c: 0.23 GB, 438,459 rows


In [17]:
# ---- 11.1c  STEP 1: 2SLS outcome equation with Mundlak means (+ comparison for step 3) ---------
# Memory-lean version: array interface instead of formulas (no extra design-matrix copies),
# one fit at a time, only the needed numbers are kept, each result is freed before the next fit.
# Coefficients / clustered SEs / first-stage F are identical to the formula version.
clusters = pd.factorize(grown_c["id"])[0]
y = grown_c["growth"]

def fit_spec(extra, iv=True):
    exog_cols = ["const"] + exog_iv + extra
    if iv:
        mod = IV2SLS(y, grown_c[exog_cols], grown_c[["log_employees_t"]], grown_c[["log_employees_tm1"]])
    else:
        mod = IV2SLS(y, grown_c[exog_cols + ["log_employees_t"]], None, None)
    return mod.fit(cov_type="clustered", clusters=clusters)

SPECS = [  # (label, extra regressors, IV?, correction term)
    ("(1) OLS, no corrections",                          [],                     False, None),
    ("(2) 2SLS only                  [= 6.4]",           [],                     True,  None),
    ("(3) OLS + Mundlak + Mills      [= 6.3]",           MEANS_A + ["mills_A"],  False, "mills_A"),
    ("(4) 2SLS + all 6.3 means + Mills [literal]",       MEANS_A + ["mills_A"],  True,  "mills_A"),
    ("(5) CRE-2SLS, no selection term",                  MEANS_B,                True,  None),
    ("(6) CRE-2SLS + Mills           [preferred]",       MEANS_B + ["mills_B"],  True,  "mills_B"),
]

rows = {}
for label, extra, iv, corr in SPECS:
    print("fitting", label.strip(), "...", ram())
    r = fit_spec(extra, iv)
    rows[label] = {
        "coef_log_emp_t": r.params["log_employees_t"],
        "se": r.std_errors["log_employees_t"],
        "p": r.pvalues["log_employees_t"],
        "first_stage_partial_F": r.first_stage.diagnostics.loc["log_employees_t", "f.stat"] if iv else np.nan,
        "mills_coef": r.params[corr] if corr else np.nan,
        "mills_p": r.pvalues[corr] if corr else np.nan,
        "n": int(r.nobs),
    }
    del r
    gc.collect()
combined_table = pd.DataFrame(rows).T

print("\nlog_employees_t coefficient -- all specifications on the SAME sample:")
display(combined_table.style.format({
    "coef_log_emp_t": "{:.4f}", "se": "{:.4f}", "p": "{:.3g}", "first_stage_partial_F": "{:,.0f}",
    "mills_coef": "{:.4f}", "mills_p": "{:.3g}", "n": "{:,}"}, na_rep="--"))

print("Reference values from the section 11 table (their own, different samples):")
for k, v in REF.items():
    print(f"  {k:<22}: {v:.4f}")

b_pref = combined_table.loc[SPECS[-1][0], "coef_log_emp_t"]
print(f"\nPreferred combined estimate (6): {b_pref:.4f}  |  distance to Mundlak within: "
      f"{abs(b_pref - REF['Mundlak within (6.3)']):.3f}, to 2SLS: {abs(b_pref - REF['2SLS (6.4)']):.3f}")

combined_table.to_csv(OUTPUT_DIR / "combined_corrections_log_employees.csv")
print("Saved:", OUTPUT_DIR / "combined_corrections_log_employees.csv")

fitting (1) OLS, no corrections ... 0.95 GB used by this notebook
fitting (2) 2SLS only                  [= 6.4] ... 1.95 GB used by this notebook
fitting (3) OLS + Mundlak + Mills      [= 6.3] ... 2.32 GB used by this notebook
fitting (4) 2SLS + all 6.3 means + Mills [literal] ... 2.43 GB used by this notebook
fitting (5) CRE-2SLS, no selection term ... 2.51 GB used by this notebook
fitting (6) CRE-2SLS + Mills           [preferred] ... 2.52 GB used by this notebook

log_employees_t coefficient -- all specifications on the SAME sample:


,coef_log_emp_t,se,p,first_stage_partial_F,mills_coef,mills_p,n
"(1) OLS, no corrections",-0.1262,0.0019,0,--,--,--,"438,459.0"
(2) 2SLS only [= 6.4],-0.0447,0.0034,0,"36,212",--,--,"438,459.0"
(3) OLS + Mundlak + Mills [= 6.3],-0.6100,0.0054,0,--,-0.0529,5.4e-05,"438,459.0"
(4) 2SLS + all 6.3 means + Mills [literal],-2.4298,0.0504,0,"1,775",-1.4014,0,"438,459.0"
"(5) CRE-2SLS, no selection term",-1.3574,0.0347,0,"1,853",--,--,"438,459.0"
(6) CRE-2SLS + Mills [preferred],-1.3595,0.0348,0,"1,846",-0.0535,0.000672,"438,459.0"


Reference values from the section 11 table (their own, different samples):
  pooled Heckman (6)    : -0.1355
  Mundlak within (6.3)  : -0.6717
  2SLS (6.4)            : -0.0447

Preferred combined estimate (6): -1.3595  |  distance to Mundlak within: 0.688, to 2SLS: 1.315
Saved: /content/growth_outputs/combined_corrections_log_employees.csv
